In [1]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re 


In [2]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [35]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        doi_link = doi_tag["href"] if doi_tag else None

        # PDF URL 생성 (DOI에서 필요한 부분만 추출하고 PDF 링크 생성)
        if doi_link:
            # DOI 링크에서 "10.18653/v1/"를 제거하고, 나머지 부분으로 PDF 링크 생성
            pdf_identifier = doi_link.split("doi.org/")[1].replace("10.18653/v1/", "")  # "/10.18653/v1/" 부분 제거
            pdf_link = f"https://aclanthology.org/{pdf_identifier}.pdf"
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url':None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [39]:
url = 'https://dblp.org/db/conf/naacl/naacl2024.html'
DB_PATH = "con_db/NAACL_conference_2024.db"
conference_name = 'NAACL 2024'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [27]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2024_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [36]:
df_papers = get_www_papers('html/NAACL_2024_accepted_papers.html', conference_name)

In [37]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Frontmatter.,,None,None,NAACL 2024
1,Named Entity Recognition Under Domain Shift vi...,"Hongyi Liu, Qingyun Wang, Payam Karisani, Heng Ji",https://aclanthology.org/2024.naacl-long.1.pdf,None,NAACL 2024
2,Text Diffusion Model with Encoder-Decoder Tran...,"Hongyi Yuan, Zheng Yuan, Chuanqi Tan, Fei Huan...",https://aclanthology.org/2024.naacl-long.2.pdf,None,NAACL 2024
3,An Interactive Framework for Profiling News Me...,"Nikhil Mehta, Dan Goldwasser",https://aclanthology.org/2024.naacl-long.3.pdf,None,NAACL 2024
4,Assessing Logical Puzzle Solving in Large Lang...,"Yinghao Li, Haorui Wang, Chao Zhang",https://aclanthology.org/2024.naacl-long.4.pdf,None,NAACL 2024


In [38]:
df_papers = df_papers.drop(index=[0])

In [40]:
save_to_database(df_papers, conference_name, DB_PATH)

487개의 논문이 NAACL 2024에 저장되었습니다.


# NAACL 2022

In [45]:
url = 'https://dblp.org/db/conf/naacl/naacl2022.html'
DB_PATH = "con_db/NAACL_conference_2022.db"
conference_name = 'NAACL 2022'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [46]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2022_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [47]:
df_papers = get_www_papers('html/NAACL_2022_accepted_papers.html', conference_name)

In [48]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Frontmatter.,,None,None,NAACL 2022
1,Social Norms Guide Reference Resolution.,"Mitchell Abrams, Matthias Scheutz",https://aclanthology.org/2022.naacl-main.1.pdf,None,NAACL 2022
2,Learning Natural Language Generation with Trun...,"Alice Martin, Guillaume Quispe, Charles Ollion...",https://aclanthology.org/2022.naacl-main.2.pdf,None,NAACL 2022
3,Language Model Augmented Monotonic Attention f...,"Sathish Reddy Indurthi, Mohd Abbas Zaidi, Beom...",https://aclanthology.org/2022.naacl-main.3.pdf,None,NAACL 2022
4,What Makes a Good and Useful Summary? Incorpor...,"Maartje ter Hoeve, Julia Kiseleva, Maarten de ...",https://aclanthology.org/2022.naacl-main.4.pdf,None,NAACL 2022


In [49]:
df_papers = df_papers.drop(index=[0])

In [50]:
save_to_database(df_papers, conference_name, DB_PATH)

442개의 논문이 NAACL 2022에 저장되었습니다.


# 2021

In [52]:
url = 'https://dblp.org/db/conf/naacl/naacl2021.html'
DB_PATH = "con_db/NAACL_conference_2021.db"
conference_name = 'NAACL 2021'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [ ]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2021_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [54]:
df_papers = get_www_papers('html/NAACL_2021_accepted_papers.html', conference_name)

In [55]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Knowledge Router: Learning Disentangled Repres...,"Shuai Zhang, Xi Rao, Yi Tay, Ce Zhang",https://aclanthology.org/2021.naacl-main.1.pdf,None,NAACL 2021
1,Distantly Supervised Relation Extraction with ...,"Fenia Christopoulou, Makoto Miwa, Sophia Anani...",https://aclanthology.org/2021.naacl-main.2.pdf,None,NAACL 2021
2,Cross-Task Instance Representation Interaction...,"Minh Van Nguyen, Viet Dac Lai, Thien Huu Nguyen",https://aclanthology.org/2021.naacl-main.3.pdf,None,NAACL 2021
3,Abstract Meaning Representation Guided Graph E...,"Zixuan Zhang, Heng Ji",https://aclanthology.org/2021.naacl-main.4.pdf,None,NAACL 2021
4,A Frustratingly Easy Approach for Entity and R...,"Zexuan Zhong, Danqi Chen",https://aclanthology.org/2021.naacl-main.5.pdf,None,NAACL 2021


In [56]:
save_to_database(df_papers, conference_name, DB_PATH)

476개의 논문이 NAACL 2021에 저장되었습니다.


# 2019

In [73]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        doi_link = doi_tag["href"] if doi_tag else None

        # PDF URL 생성 (DOI에서 필요한 부분만 추출하고 PDF 링크 생성)
        if doi_link:
            # DOI 링크에서 "10.18653/v1/"를 제거하고, 나머지 부분으로 PDF 링크 생성
            pdf_identifier = doi_link.split("doi.org/")[1].replace("10.18653/v1/", "")  # "/10.18653/v1/" 부분 제거
            pdf_link = f"https://aclanthology.org/{pdf_identifier}.pdf"
            
            # 'n'을 'N'으로 바꿔서 정확히 'n19' -> 'N19'로만 수정
            pdf_link = pdf_link.replace('n19', 'N19', 1)
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [97]:
url = 'https://dblp.org/db/conf/naacl/naacl2019-6.html'
DB_PATH = "con_db/NAACL_conference_2019.db"
conference_name = 'NAACL 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [98]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [99]:
df_papers = get_www_papers('html/NAACL_2019_accepted_papers.html', conference_name)

In [100]:
df_papers.head()

""


In [96]:
save_to_database(df_papers,conference_name,DB_PATH)

6개의 논문이 NAACL 2019에 저장되었습니다.


# 2018

In [111]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        doi_link = doi_tag["href"] if doi_tag else None

        # PDF URL 생성 (DOI에서 필요한 부분만 추출하고 PDF 링크 생성)
        if doi_link:
            # DOI 링크에서 "10.18653/v1/"를 제거하고, 나머지 부분으로 PDF 링크 생성
            pdf_identifier = doi_link.split("doi.org/")[1].replace("10.18653/v1/", "")  # "/10.18653/v1/" 부분 제거
            pdf_link = f"https://aclanthology.org/{pdf_identifier}.pdf"
            
            # 'n'을 'N'으로 바꿔서 정확히 'n19' -> 'N19'로만 수정
            pdf_link = pdf_link.replace('n18', 'N18', 1)
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [144]:
url = 'https://dblp.org/db/conf/naacl/naacl2018-6.html'
DB_PATH = "con_db/NAACL_conference_2018.db"
conference_name = 'NAACL 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [145]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [146]:
df_papers = get_www_papers('html/NAACL_2018_accepted_papers.html', conference_name)

In [147]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,"Modelling Natural Language, Programs, and thei...","Graham Neubig, Miltiadis Allamanis",https://aclanthology.org/N18-6001.pdf,None,NAACL 2018
1,Deep Learning Approaches to Text Production.,"Claire Gardent, Shashi Narayan",https://aclanthology.org/N18-6002.pdf,None,NAACL 2018
2,Scalable Construction and Reasoning of Massive...,"Xiang Ren, Nanyun Peng, William Yang Wang",https://aclanthology.org/N18-6003.pdf,None,NAACL 2018
3,The interplay between lexical resources and Na...,"José Camacho-Collados, Luis Espinosa Anke, Moh...",https://aclanthology.org/N18-6004.pdf,None,NAACL 2018
4,Socially Responsible NLP.,"Yulia Tsvetkov, Vinodkumar Prabhakaran, Rob Voigt",https://aclanthology.org/N18-6005.pdf,None,NAACL 2018


In [139]:
save_to_database(df_papers, conference_name,DB_PATH)

6개의 논문이 NAACL 2018에 저장되었습니다.


# 2016

In [175]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        doi_link = doi_tag["href"] if doi_tag else None

        # PDF URL 생성 (DOI에서 필요한 부분만 추출하고 PDF 링크 생성)
        if doi_link:
            # DOI 링크에서 "10.18653/v1/"를 제거하고, 나머지 부분으로 PDF 링크 생성
            pdf_identifier = doi_link.split("doi.org/")[1].replace("10.18653/v1/", "")  # "/10.18653/v1/" 부분 제거
            pdf_link = f"https://aclanthology.org/{pdf_identifier}.pdf"
            
            # 'n'을 'N'으로 바꿔서 정확히 'n19' -> 'N19'로만 수정
            pdf_link = pdf_link.replace('w16', 'W16', 1)
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [199]:
url = 'https://dblp.org/db/conf/naacl/sedmt2016.html'
DB_PATH = "con_db/NAACL_conference_2016.db"
conference_name = 'NAACL 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [200]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [201]:
df_papers = get_www_papers('html/NAACL_2016_accepted_papers.html', conference_name)

In [202]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Deterministic natural language generation from...,Alastair Butler,https://aclanthology.org/W16-0601.pdf,None,NAACL 2016
1,Extending Phrase-Based Translation with Depend...,"Liangyou Li, Andy Way, Qun Liu",https://aclanthology.org/W16-0602.pdf,None,NAACL 2016
2,The Naming Sharing Structure and its Cognitive...,"Shi-Li Ge, Rou Song",https://aclanthology.org/W16-0603.pdf,None,NAACL 2016
3,Towards Semantic-based Hybrid Machine Translat...,"Kiril Ivanov Simov, Petya Osenova, Alexander P...",https://aclanthology.org/W16-0604.pdf,None,NAACL 2016


In [203]:
save_to_database(df_papers, conference_name, DB_PATH)

4개의 논문이 NAACL 2016에 저장되었습니다.


# 2015

In [221]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        doi_link = doi_tag["href"] if doi_tag else None

        # PDF URL 생성 (DOI에서 필요한 부분만 추출하고 PDF 링크 생성)
        if doi_link:
            # DOI 링크에서 "10.18653/v1/"를 제거하고, 나머지 부분으로 PDF 링크 생성
            pdf_identifier = doi_link.split("doi.org/")[1].replace("10.3115/v1/", "")  # "/10.18653/v1/" 부분 제거
            pdf_link = f"https://aclanthology.org/{pdf_identifier}.pdf"
            
            # 'n'을 'N'으로 바꿔서 정확히 'n19' -> 'N19'로만 수정
            pdf_link = pdf_link.replace('w15', 'W15', 1)
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [252]:
url = 'https://dblp.org/db/conf/ssst/ssst2015.html'
DB_PATH = "con_db/NAACL_conference_2015.db"
conference_name = 'NAACL 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [253]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [254]:
df_papers = get_www_papers('html/NAACL_2015_accepted_papers.html', conference_name)

In [255]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Harmonizing word alignments and syntactic stru...,"Dun Deng, Nianwen Xue, Shiman Guo",https://aclanthology.org/W15-1001.pdf,None,NAACL 2015
1,Non-projective Dependency-based Pre-Reordering...,"Antonio Valerio Miceli Barone, Giuseppe Attardi",https://aclanthology.org/W15-1002.pdf,None,NAACL 2015
2,"Translating Negation: Induction, Search And Mo...","Federico Fancellu, Bonnie L. Webber",https://aclanthology.org/W15-1003.pdf,None,NAACL 2015
3,"SMT error analysis and mapping to syntactic, s...",Nora Aranberri,https://aclanthology.org/W15-1004.pdf,None,NAACL 2015
4,Unsupervised False Friend Disambiguation Using...,"Maryam Aminian, Mahmoud Ghoneim, Mona T. Diab",https://aclanthology.org/W15-1005.pdf,None,NAACL 2015


In [256]:
save_to_database(df_papers, conference_name, DB_PATH)

12개의 논문이 NAACL 2015에 저장되었습니다.


# 2013

In [288]:
import pandas as pd
from bs4 import BeautifulSoup
import re

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # PDF URL 생성 (aclanthology.org 형식으로 링크를 찾고, .pdf를 추가)
        pdf_tag = entry.find("a", href=re.compile(r"aclanthology\.org"))
        if pdf_tag:
            # 'aclanthology.org' 링크에서 논문 ID를 추출하고, '.pdf'를 추가
            pdf_identifier = pdf_tag["href"].split("/")[-2]  # 링크에서 {ID} 부분 추출
            pdf_link = f"https://aclanthology.org/{pdf_identifier}.pdf"
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [324]:
url = 'https://dblp.org/db/conf/naacl/events2013.html'
DB_PATH = "con_db/NAACL_conference_2013.db"
conference_name = 'NAACL 2013'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [325]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NAACL_2013_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [326]:
df_papers = get_www_papers('html/NAACL_2013_accepted_papers.html', conference_name)

In [327]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Coping With Implicit Arguments And Events Core...,Rodolfo Delmonte,https://aclanthology.org/W13-1201.pdf,None,NAACL 2013
1,GAF: A Grounded Annotation Framework for Events.,"Antske Fokkens, Marieke van Erp, Piek Vossen, ...",https://aclanthology.org/W13-1202.pdf,None,NAACL 2013
2,"Events are Not Simple: Identity, Non-Identity,...","Eduard H. Hovy, Teruko Mitamura, Felisa Verdej...",https://aclanthology.org/W13-1203.pdf,None,NAACL 2013
3,Event representation across genre.,"Lidia Pivovarova, Silja Huttunen, Roman Yangarber",https://aclanthology.org/W13-1204.pdf,None,NAACL 2013
4,A Semantic Tool for Historical Events.,Ryan Shaw,https://aclanthology.org/W13-1205.pdf,None,NAACL 2013


In [304]:
df_papers = df_papers.drop(index=[0])

In [328]:
save_to_database(df_papers, conference_name, DB_PATH)

6개의 논문이 NAACL 2013에 저장되었습니다.
